In [ ]:
# ============================================================
# 01_preprocessing.ipynb
# KNHANES 2020-2024 preprocessing for sex-stratified diabetes
# risk modelling in insurance underwriting.
#
# Input : data/raw/hn20_all.sas7bdat ... hn24_all.sas7bdat
#         (Korea National Health and Nutrition Examination Survey,
#          KDCA; not redistributed here -- see README)
# Output: data/processed/df_total.csv, df_male.csv, df_female.csv
#
# Pipeline: variable selection -> 5-year merge -> target recoding
#           -> adult (>=19) filter -> complete-case -> type casting
#           -> 3SD outlier removal -> sex split.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat
import warnings
warnings.filterwarnings("ignore")

# ---- Resolve repository root (works from notebooks/ or repo root) ----
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# ---- Variable definitions ----
key = ["ID", "year", "sex"]
cat_col = [
    "HE_obe", "BO1_1", "BO1_2", "BO1_3", "BD1_11", "BD2_1", "BS3_1",
    "BE3_71", "BE3_75", "BE3_81", "BE3_91", "pa_aerobic", "L_BR_FQ",
    "BP1", "mh_stress", "incm", "ho_incm", "edu", "BH1",
]
num_col = ["HE_BMI", "HE_wc", "HE_wt", "N_SUGAR", "N_CHO", "N_EN", "age"]
target_col = ["HE_DM_HbA1c"]  # original: 1=Normal, 3=Diabetes
all_vars = key + cat_col + num_col + target_col
print(f"Selected variables: {len(all_vars)} "
      f"(key {len(key)}, categorical {len(cat_col)}, "
      f"continuous {len(num_col)}, target {len(target_col)})")

# ---- Load raw SAS files (2020-2024) ----
frames = []
for y in (20, 21, 22, 23, 24):
    df, _ = pyreadstat.read_sas7bdat(str(RAW / f"hn{y}_all.sas7bdat"))
    frames.append(df[all_vars].copy())
    print(f"  20{y}: {df.shape[0]:,} rows loaded")

# ---- Merge 5 survey years ----
df = pd.concat(frames, axis=0).reset_index(drop=True)
print(f"Merged: {df.shape}")

# ---- Target recoding: 1->0 (non-diabetic), 3->1 (diabetic); else NaN ----
df["HE_DM_HbA1c"] = df["HE_DM_HbA1c"].map({1: 0, 3: 1})

# ---- Keep valid target + adults (>=19) ----
df = df[df["HE_DM_HbA1c"].isin([0, 1])].reset_index(drop=True)
df = df[df["age"] >= 19].reset_index(drop=True)
print(f"After valid-target + adult filter: {df.shape[0]:,} rows")

# ---- Complete-case analysis (listwise deletion) ----
df = df.dropna(subset=all_vars).reset_index(drop=True)
print(f"After complete-case analysis: {df.shape[0]:,} rows")

# ---- Type casting ----
df[target_col] = df[target_col].astype(int)
for c in cat_col:
    df[c] = df[c].astype(int)
for c in num_col:
    df[c] = df[c].astype(float)

# ---- Outlier removal: mean +/- 3SD per continuous variable ----
before = len(df)
for c in num_col:
    m, s = df[c].mean(), df[c].std()
    lo, hi = m - 3 * s, m + 3 * s
    df = df[(df[c] >= lo) & (df[c] <= hi)].reset_index(drop=True)
print(f"After 3SD outlier removal: {len(df):,} rows (removed {before - len(df):,})")

# ---- Sex split (1=male, 2=female) ----
df_male = df[df["sex"] == 1].reset_index(drop=True)
df_female = df[df["sex"] == 2].reset_index(drop=True)

# ---- Persist processed datasets ----
df.to_csv(PROC / "df_total.csv", index=False)
df_male.to_csv(PROC / "df_male.csv", index=False)
df_female.to_csv(PROC / "df_female.csv", index=False)

# ---- Summary ----
def summ(name, d):
    n, pos = len(d), int(d["HE_DM_HbA1c"].sum())
    print(f"  {name:7s}: N={n:,}, diabetic={pos:,} ({pos/n*100:.1f}%)")
print("Final datasets written to data/processed/:")
summ("total", df); summ("male", df_male); summ("female", df_female)

# ---- Feature list for downstream notebooks (exclude identifiers/target) ----
features = cat_col + num_col
pd.Series(features, name="feature").to_csv(PROC / "feature_list.csv", index=False)
print(f"Feature list ({len(features)} predictors) saved.")
